# Solo Enterprise for Istio, ambient, in two parts: multicluster and workload identity

Two kind clusters (`mesh1` + `mesh2`) running Solo ambient, peered over HBONE, stood up by **one script** (`./setup.sh`). This notebook is the demo on top: every command and manifest is in the cell, nothing hidden behind a script.

- **Part 1 — Multicluster.** Bookinfo on both clusters, east-west gateways, peering, **global services** (`*.mesh.internal`), cross-cluster **failover** and **takeover**, and an L7 **waypoint**.
- **Part 2 — Workload identity (L4/L7).** The petshop identity story on `mesh1` only: the SVID *is* the identity, L4 authorization in ztunnel, the shared-ServiceAccount gap, **workload claims** closing it, then the agentgateway waypoint for JWT, CEL authorization, canary routing and identity-keyed rate limiting.

**The parts are independent** — run Part 1 or Part 2 in isolation (both only need `./setup.sh` to be green). Open the Gloo UI first (`./demo-scripts/consoles.sh`) — its service graph spans both clusters.

> **Kernel:** Bash (Select Kernel → Jupyter Kernel → **Bash**).
> After a laptop sleep, heal expired mesh certs with `./demo-scripts/wake.sh`.

In [ ]:
# Connect — contexts, the Solo istioctl build, licences. Run this first, always.
[ -d istio-ambient-demo-kind ] && cd istio-ambient-demo-kind || :
export CLUSTER1=kind-mesh1 CLUSTER2=kind-mesh2
export ISTIOCTL=$HOME/.istioctl/bin/istioctl-1.30.3-solo
export SECRETS_FILE="${SECRETS_FILE:-$HOME/code/solo/secrets/secrets-envs.sh}"
[ -f "$SECRETS_FILE" ] && set -a && . "$SECRETS_FILE" && set +a
echo "mesh1=$CLUSTER1  mesh2=$CLUSTER2  licence: $([ -n "$SOLO_ISTIO_LICENSE_KEY" ] && echo yes || echo NO)"
$ISTIOCTL --context $CLUSTER1 multicluster check 2>&1 | grep -E "Peers Check|Gateway Check" || echo "not peered — run ./setup.sh (or ./demo-scripts/wake.sh after a sleep)"


---
# Part 1 · Multicluster ambient

Two clusters, one mesh. `setup.sh` installed Solo ambient on both with a **shared root CA** and per-cluster intermediates, so the clusters already share a root of trust. This part deploys bookinfo on both, walks the peering (east-west gateways + link), makes `reviews` a **global service**, kills it locally to show **cross-cluster failover**, takes over the local hostname, and finishes with an L7 **waypoint**.

Cluster layout (matching the deck): `mesh1` runs reviews **v1+v2** (black stars), `mesh2` also runs **v3** (red stars) — so the moment traffic is served cross-cluster, you can *see* it.

## 1.1 · Bookinfo on both clusters — and enrol it in the mesh

Two labels on the namespace do all the work:
- `istio.io/dataplane-mode=ambient` — enrols every pod (ztunnel captures its traffic; no restarts, no sidecars)
- `topology.istio.io/network=<cluster>` — tells istiod which network the pods live on, so it can rewrite endpoints across clusters

In [ ]:
for CTX in $CLUSTER1 $CLUSTER2; do
  kubectl --context $CTX create namespace bookinfo --dry-run=client -o yaml | kubectl --context $CTX apply -f -
  kubectl --context $CTX label namespace bookinfo \
    istio.io/dataplane-mode=ambient \
    topology.istio.io/network=${CTX#kind-} --overwrite
done


In [ ]:
BOOK=https://raw.githubusercontent.com/istio/istio/release-1.24/samples/bookinfo/platform/kube
for CTX in $CLUSTER1 $CLUSTER2; do
  kubectl --context $CTX -n bookinfo apply -f $BOOK/bookinfo.yaml
  kubectl --context $CTX -n bookinfo apply -f $BOOK/bookinfo-versions.yaml
done
# match the deck: mesh1 runs v1+v2 only; mesh2 keeps v3 (red stars) as well
kubectl --context $CLUSTER1 -n bookinfo delete deploy reviews-v3 --ignore-not-found

# a client pod on mesh1 to curl from
kubectl --context $CLUSTER1 -n bookinfo apply -f - <<'EOF'
apiVersion: apps/v1
kind: Deployment
metadata: { name: demo-client, namespace: bookinfo, labels: { app: demo-client } }
spec:
  replicas: 1
  selector: { matchLabels: { app: demo-client } }
  template:
    metadata: { labels: { app: demo-client } }
    spec:
      containers:
        - name: curl
          image: curlimages/curl:8.14.1
          command: ["sleep", "infinity"]
EOF

kubectl --context $CLUSTER1 -n bookinfo wait --for=condition=Ready pod --all --timeout=240s
kubectl --context $CLUSTER2 -n bookinfo wait --for=condition=Ready pod --all --timeout=240s


## 1.2 · Peer the clusters — east-west gateways + link

Peering is **two istioctl commands** (the Solo build). `expose` creates the east-west gateway (an HBONE listener on `:15008` + istiod xDS on `:15012`, on a LoadBalancer IP); `link` creates `istio-remote` Gateways in each cluster pointing at the peer's east-west address. No remote kubeconfig secrets — the Solo distribution peers istiod-to-istiod over xDS (`DISABLE_LEGACY_MULTICLUSTER`), a decentralised push model.

`setup.sh` already ran these, so they no-op here — run them anyway to show *how* the clusters were linked.

In [ ]:
# east-west gateway on each cluster (idempotent)
for CTX in $CLUSTER1 $CLUSTER2; do
  $ISTIOCTL --context $CTX multicluster expose -n istio-eastwest
done
# link them bidirectionally (idempotent)
$ISTIOCTL multicluster link --namespace istio-eastwest --contexts=$CLUSTER1,$CLUSTER2


In [ ]:
# what link created: the local east-west gateway + a remote peer gateway per cluster
kubectl --context $CLUSTER1 -n istio-eastwest get gateways
kubectl --context $CLUSTER2 -n istio-eastwest get gateways
echo
$ISTIOCTL --context $CLUSTER1 multicluster check


## 1.3 · Reach the app — agentgateway ingress

North-south is **Solo Enterprise for agentgateway** (`GatewayClass: enterprise-agentgateway`) — a standard Gateway API data plane. One `Gateway` + one `HTTPRoute`, and productpage is on a LoadBalancer IP. **Open the printed URL in your browser and keep the tab open** — you'll watch the stars change during failover.

In [ ]:
kubectl --context $CLUSTER1 -n bookinfo apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata: { name: bookinfo-gateway, namespace: bookinfo }
spec:
  gatewayClassName: enterprise-agentgateway
  listeners:
  - { name: http, port: 8080, protocol: HTTP, allowedRoutes: { namespaces: { from: Same } } }
---
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: { name: productpage, namespace: bookinfo }
spec:
  parentRefs: [{ name: bookinfo-gateway }]
  rules:
  - backendRefs: [{ name: productpage, port: 9080 }]
EOF
kubectl --context $CLUSTER1 -n bookinfo wait --for=condition=Programmed gateway/bookinfo-gateway --timeout=90s
# the agentgateway data-plane pod is created ON DEMAND, a few seconds AFTER the
# Gateway goes Programmed — poll for it before waiting Ready (kubectl wait errors
# on "no matching resources" if the pod does not exist yet)
for i in $(seq 1 40); do
  kubectl --context $CLUSTER1 -n bookinfo get pod -l gateway.networking.k8s.io/gateway-name=bookinfo-gateway 2>/dev/null | grep -q . && break
  sleep 3
done
kubectl --context $CLUSTER1 -n bookinfo wait --for=condition=Ready \
  pod -l gateway.networking.k8s.io/gateway-name=bookinfo-gateway --timeout=120s

GW=$(kubectl --context $CLUSTER1 -n bookinfo get gateway bookinfo-gateway -o jsonpath='{.status.addresses[0].value}')
echo "open →  http://$GW:8080/productpage"
# LB IP + pod may need a moment to answer — retry until the first 200
for i in $(seq 1 20); do
  code=$(curl -s -o /dev/null -w '%{http_code}' -m 5 "http://$GW:8080/productpage")
  [ "$code" = "200" ] && break; sleep 3
done
echo "HTTP $code"


## 1.4 · Canary + rate limit at the gateway

The same gateway is a full L7 router. Weight `/reviews` 80/20 between v1 and v2, then attach a local rate limit to the productpage route — both are plain Gateway API + one policy, no app change.

In [ ]:
kubectl --context $CLUSTER1 -n bookinfo apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: { name: reviews-canary, namespace: bookinfo }
spec:
  parentRefs: [{ name: bookinfo-gateway }]
  rules:
  - matches: [{ path: { type: PathPrefix, value: /reviews } }]
    backendRefs:
    - { name: reviews-v1, port: 9080, weight: 80 }
    - { name: reviews-v2, port: 9080, weight: 20 }
EOF
sleep 3
GW=$(kubectl --context $CLUSTER1 -n bookinfo get gateway bookinfo-gateway -o jsonpath='{.status.addresses[0].value}')
for i in $(seq 1 20); do curl -s "http://$GW:8080/reviews/0" | grep -oE 'reviews-v[0-9]'; done | sort | uniq -c


In [ ]:
kubectl --context $CLUSTER1 -n bookinfo apply -f - <<'EOF'
apiVersion: agentgateway.dev/v1alpha1
kind: AgentgatewayPolicy
metadata: { name: ingress-ratelimit, namespace: bookinfo }
spec:
  targetRefs:
  - { group: gateway.networking.k8s.io, kind: HTTPRoute, name: productpage }
  traffic:
    rateLimit:
      local:
      - requests: 2
        unit: Seconds
EOF
sleep 4
GW=$(kubectl --context $CLUSTER1 -n bookinfo get gateway bookinfo-gateway -o jsonpath='{.status.addresses[0].value}')
for i in $(seq 1 8); do curl -s -o /dev/null -w "%{http_code} " "http://$GW:8080/productpage"; done; echo
# → first 2 pass (200), the rest 429


## 1.5 · Global services — one hostname, endpoints in both clusters

**One label** makes a service global: `solo.io/service-scope=global`. Solo Istio then publishes `reviews.bookinfo.mesh.internal` — a mesh-wide hostname whose endpoints span every cluster that labels the service — and istiod auto-generates the ServiceEntry + VIP. The original `reviews.bookinfo.svc.cluster.local` stays **local-only**: nothing changes for existing callers.

Traffic to the global hostname prefers **local** endpoints (`PreferNetwork` is the default for global services) and spills to the peer only when local is gone. That is the resilience story: same hostname, traffic stays local until it can't.

In [ ]:
for CTX in $CLUSTER1 $CLUSTER2; do
  kubectl --context $CTX -n bookinfo label svc reviews solo.io/service-scope=global --overwrite
done
sleep 8
# istiod generated the global hostname automatically:
kubectl --context $CLUSTER1 -n istio-system get serviceentry
$ISTIOCTL --context $CLUSTER1 multicluster check 2>/dev/null | grep -iA2 "shared services" || true


In [ ]:
# mesh1 still has local reviews pods, so calls to the GLOBAL hostname stay LOCAL
echo "--- 10 calls to reviews.bookinfo.mesh.internal from mesh1 (expect only mesh1 pods) ---"
for i in $(seq 1 10); do
  kubectl --context $CLUSTER1 -n bookinfo exec deploy/demo-client -- \
    curl -s -m 6 http://reviews.bookinfo.mesh.internal:9080/reviews/0 2>/dev/null | grep -oE '"podname": "[^"]+"'
done | sort | uniq -c
echo
echo "mesh1 reviews pods:"; kubectl --context $CLUSTER1 -n bookinfo get pods -l app=reviews -o name


## 1.6 · Kill it locally — cross-cluster failover, live

Scale mesh1's `reviews` to **zero**. Watch two hostnames diverge — this is the deck's global-vs-local table, live:

| hostname | result |
|---|---|
| `reviews.bookinfo.mesh.internal` (global) | keeps serving — endpoints fail over to `mesh2`, over HBONE through the east-west gateways |
| `reviews.bookinfo.svc.cluster.local` (local) | breaks — the label left it local-only, and there is nothing local left |

**Check the browser tab**: productpage calls the *local* hostname, so its reviews panel breaks too. The next cell fixes exactly that.

In [ ]:
for v in v1 v2; do kubectl --context $CLUSTER1 -n bookinfo scale deploy reviews-$v --replicas=0; done
kubectl --context $CLUSTER1 -n bookinfo wait --for=delete pod -l app=reviews --timeout=60s
sleep 8

echo "--- GLOBAL hostname: still serving, now from mesh2 (expect v3 pods too) ---"
for i in $(seq 1 6); do
  kubectl --context $CLUSTER1 -n bookinfo exec deploy/demo-client -- \
    curl -s -m 6 http://reviews.bookinfo.mesh.internal:9080/reviews/0 2>/dev/null | grep -oE '"podname": "[^"]+"'
done | sort | uniq -c

echo
echo "--- LOCAL hostname: local-only, and local is gone ---"
kubectl --context $CLUSTER1 -n bookinfo exec deploy/demo-client -- \
  curl -s -o /dev/null -w "reviews.bookinfo.svc.cluster.local -> %{http_code}\n" -m 5 \
  http://reviews.bookinfo.svc.cluster.local:9080/reviews/0 || true
# refresh the productpage tab: the reviews panel is down (it calls the local hostname)


## 1.7 · Takeover — the local hostname fails over too

Apps like productpage call `reviews.bookinfo.svc.cluster.local` and you are not going to rewrite them. **`solo.io/service-takeover=true`** (Solo Istio 1.27.2+) redirects the *local* hostname onto the global endpoint set: same app, same hostname, now served by `mesh2`. (`solo.io/service-scope=global-only` is the older single-label spelling of the same behaviour.)

**Refresh the browser tab** after the cell: the reviews panel is back — with **red stars**, because it is being served by mesh2's v3. Cross-cluster traffic you can see.

In [ ]:
kubectl --context $CLUSTER1 -n bookinfo label svc reviews solo.io/service-takeover=true --overwrite
sleep 10

echo "--- LOCAL hostname, taken over: served cross-cluster by mesh2 ---"
for i in $(seq 1 6); do
  kubectl --context $CLUSTER1 -n bookinfo exec deploy/demo-client -- \
    curl -s -m 6 http://reviews.bookinfo.svc.cluster.local:9080/reviews/0 2>/dev/null | grep -oE '"podname": "[^"]+"'
done | sort | uniq -c
GW=$(kubectl --context $CLUSTER1 -n bookinfo get gateway bookinfo-gateway -o jsonpath='{.status.addresses[0].value}')
echo
echo "refresh →  http://$GW:8080/productpage   (reviews back — red stars = mesh2's v3)"


**Gloo UI moment.** Open the Graph (both clusters ticked, `bookinfo` namespace): the edge from mesh1's productpage now crosses into mesh2's reviews — the cross-cluster hop drawn live. Drop the time window to the smallest setting so it reshapes quickly.

Now restore mesh1 and watch locality reassert — traffic returns local (black stars), cross-cluster only during the outage.

In [ ]:
for v in v1 v2; do kubectl --context $CLUSTER1 -n bookinfo scale deploy reviews-$v --replicas=1; done
kubectl --context $CLUSTER1 -n bookinfo wait --for=condition=Ready pod -l app=reviews --timeout=120s
sleep 12
echo "--- local endpoints are back: traffic returns LOCAL (PreferNetwork) ---"
for i in $(seq 1 8); do
  kubectl --context $CLUSTER1 -n bookinfo exec deploy/demo-client -- \
    curl -s -m 6 http://reviews.bookinfo.svc.cluster.local:9080/reviews/0 2>/dev/null | grep -oE '"podname": "[^"]+"'
done | sort | uniq -c


## 1.8 · L7 inside the mesh — the waypoint

Everything so far at L7 happened at the ingress. For **service-to-service** L7 (routing, headers, policy between meshed workloads) ambient adds a **waypoint** — and on Solo Enterprise the waypoint data plane is **agentgateway** (`GatewayClass: enterprise-agentgateway-waypoint`), not Envoy. Enrol the namespace onto it with one label, then attach a Gateway API `HTTPRoute` whose `parentRef` is the reviews **Service** — mesh routing, no ingress involved: productpage's own calls to `reviews` now honour the split.

**Watch the browser**: reviews pins to **v2 (black stars)** for every refresh.

In [ ]:
kubectl --context $CLUSTER1 apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata:
  name: bookinfo-waypoint
  namespace: bookinfo
  labels: { istio.io/waypoint-for: service }
spec:
  gatewayClassName: enterprise-agentgateway-waypoint
  listeners:
  - name: mesh
    port: 15088
    protocol: HTTP
EOF
kubectl --context $CLUSTER1 label namespace bookinfo istio.io/use-waypoint=bookinfo-waypoint --overwrite
kubectl --context $CLUSTER1 -n bookinfo rollout status deploy/bookinfo-waypoint --timeout=150s

# route AT the waypoint: parentRef is the reviews SERVICE (GAMMA) — pin all traffic to v2
kubectl --context $CLUSTER1 -n bookinfo apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: { name: reviews-mesh-split, namespace: bookinfo }
spec:
  parentRefs:
    - group: ""
      kind: Service
      name: reviews
      port: 9080
  rules:
    - backendRefs:
        - { name: reviews-v2, port: 9080, weight: 100 }
EOF
sleep 8
echo "--- 8 mesh calls to reviews (via the waypoint): all v2 ---"
for i in $(seq 1 8); do
  kubectl --context $CLUSTER1 -n bookinfo exec deploy/demo-client -- \
    curl -s -m 6 http://reviews:9080/reviews/0 2>/dev/null | grep -oE 'reviews-v[0-9]'
done | sort | uniq -c
# proof the waypoint carried it: agentgateway logs every request
kubectl --context $CLUSTER1 -n bookinfo logs deploy/bookinfo-waypoint --tail=3


## 1.R · Reset Part 1

Puts bookinfo back to steady state (keeps the app + ingress + global label, removes the waypoint, canary, rate limit and takeover) so Part 1 can be re-run — or so Part 2 starts clean.

In [ ]:
kubectl --context $CLUSTER1 label namespace bookinfo istio.io/use-waypoint- 2>/dev/null || true
kubectl --context $CLUSTER1 -n bookinfo delete httproute reviews-mesh-split reviews-canary --ignore-not-found
kubectl --context $CLUSTER1 -n bookinfo delete gateway bookinfo-waypoint --ignore-not-found
kubectl --context $CLUSTER1 -n bookinfo delete agentgatewaypolicy ingress-ratelimit --ignore-not-found
kubectl --context $CLUSTER1 -n bookinfo label svc reviews solo.io/service-takeover- 2>/dev/null || true
for v in v1 v2; do kubectl --context $CLUSTER1 -n bookinfo scale deploy reviews-$v --replicas=1; done
echo "Part 1 reset."


---
# Part 2 · Workload identity — the certificate is the identity

Runs on **`mesh1` only** (no peering needed — this part stands alone). One app, a petshop, and the whole ambient security model: SPIFFE identity, L4 authorization in ztunnel, identity-aware access logs, workload claims, then an **agentgateway waypoint** for L7 (JWT + CEL + canary + rate limiting).

The trust domain on `mesh1` is **`mesh1`** (set as a Helm value at install — not `cluster.local`), so identities are `spiffe://mesh1/ns/petshop/sa/<sa>`. The cast:

| Workload | ServiceAccount → identity | Role |
|---|---|---|
| `petstore` | `sa/petstore` | the API — `GET /pets`, `DELETE /pets/{id}` |
| `storefront` | `sa/storefront` | client the L4 policy **allows** |
| `analytics` | `sa/analytics` | client the L4 policy **denies** |
| `checkout-blue` | `sa/checkout` | shares one SA with green … |
| `checkout-green` | `sa/checkout` | … so both present the **same** SVID |

Each client curls `petstore:8080/pets` every 2 seconds and prints the HTTP code — the observe-cells read the client's last log line, which is why each policy step sleeps briefly first. A denied client shows `000` (connection reset), an allowed one `200`.

In [ ]:
# Part 2 context — independent of Part 1. Run this first (after Connect).
[ -d istio-ambient-demo-kind ] && cd istio-ambient-demo-kind || :
export CTX=kind-mesh1 ISTIO_NS=istio-system TD=mesh1
export ISTIOCTL=$HOME/.istioctl/bin/istioctl-1.30.3-solo
export HUB=us-docker.pkg.dev/soloio-img/istio TAG=1.30.3-solo
export HREPO=oci://us-docker.pkg.dev/soloio-img/istio-helm HVER=1.30.3-solo
export SECRETS_FILE="${SECRETS_FILE:-$HOME/code/solo/secrets/secrets-envs.sh}"
[ -f "$SECRETS_FILE" ] && set -a && . "$SECRETS_FILE" && set +a
echo "context: $CTX ; trust domain: $TD ; licence: $([ -n "$SOLO_ISTIO_LICENSE_KEY" ] && echo yes || echo NO)"


## 2.1 · The petshop app

One namespace, enrolled in ambient with a single label — no restarts, no sidecars.

In [ ]:
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: v1
kind: Namespace
metadata:
  name: petshop
  labels:
    istio.io/dataplane-mode: ambient
EOF

kubectl --context $CTX apply -f yaml/10-petshop/
kubectl --context $CTX -n petshop rollout status deploy/petstore deploy/storefront deploy/analytics deploy/checkout-blue deploy/checkout-green --timeout=180s
kubectl --context $CTX -n petshop get pods


## 2.2 · Identity is the certificate

ztunnel holds one mTLS SVID per workload **identity**, and the identity is the ServiceAccount — the URI SAN is exactly what an L4 policy matches on. So look for what is *missing* below: five pods, but only **four** leaf certs. `checkout-blue` and `checkout-green` never appear by name; they share `sa/checkout`, so ztunnel presents **one** cert for both. That is the ceiling §2.5 hits, and §2.7 removes.

In [ ]:
# five pods, but their identity is the ServiceAccount…
kubectl --context $CTX -n petshop get pods \
  -o custom-columns=POD:.metadata.name,SERVICEACCOUNT:.spec.serviceAccountName

# …so the ztunnel on their node holds only FOUR leaf certs — checkout appears once
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=mesh1-worker -o jsonpath='{.items[0].metadata.name}')
$ISTIOCTL --context $CTX ztunnel-config certificates "$ZT.$ISTIO_NS" | grep -E "CERTIFICATE NAME|petshop" | grep -Ei "name|leaf"


## 2.3 · Authorize on identity, at L4

One `AuthorizationPolicy`, enforced by ztunnel, no waypoint. It selects `petstore` and allows only the `storefront` identity — so it simultaneously **allows storefront** and **denies analytics and checkout** (ztunnel fails closed: once any ALLOW selects a workload, everything unnamed is denied).

Principals use the trust domain **`mesh1`** — a `cluster.local/...` principal here would match nothing.

In [ ]:
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: allow-storefront, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - from: [{ source: { principals: ["mesh1/ns/petshop/sa/storefront"] } }]
    to:   [{ operation: { ports: ["8080"] } }]
EOF
sleep 10
# each client curls petstore every 2s and prints the HTTP code
for d in storefront analytics checkout-blue checkout-green; do
  echo "$d: $(kubectl --context $CTX -n petshop logs deploy/$d --tail=1)"
done


**Watching in the Gloo UI and the denied workloads still show edges to `petstore`?** The graph is drawn from **completed-request telemetry** over a rolling window — those edges are pre-policy 200s still ageing out. A workload denied at L4 never completes a request, so once its history rolls out of the window it draws nothing. Drop the Graph's time-interval picker to the smallest window to make it reshape within a minute or two. The per-connection allow/deny verdicts are in ztunnel's access logs — next.

## 2.4 · Read the L4 access logs

ztunnel logs every connection with the peer SPIFFE identities and the outcome — identity-aware audit at L4, no waypoint. With the `allow-storefront` policy in force you'll see `storefront` **ALLOW** and `analytics` **DENY**, each tagged with its `src.identity`.

In [ ]:
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=mesh1-worker -o jsonpath='{.items[0].metadata.name}')
kubectl --context $CTX -n $ISTIO_NS logs "$ZT" --tail=400 \
 | grep '"scope":"access"' | grep petshop | tail -8 \
 | jq -rc '{src:(.["src.identity"]//"-"),
            dst:(.["dst.service"]//.["dst.identity"]//"-"),
            dir:(.direction//"-"),
            result:(if (.error//"")=="" then "ALLOW" else ("DENY: "+(.error|tostring)) end)}'


## 2.5 · The shared-ServiceAccount ceiling

Add `sa/checkout` to the allowed set. Both `checkout-blue` and `checkout-green` get in — and there is **no L4 rule that lets one in and keeps the other out**. The identity *is* the certificate, the certificate is issued per ServiceAccount, and both pods present the same `sa/checkout` SVID. To ztunnel they are literally the same caller.

Blue/green makes the problem easy to see, but the everyday version is worse: any pod that never sets `serviceAccountName` runs as the namespace's **`default`** ServiceAccount — write an L4 ALLOW for "the payments app" and you have admitted every workload in that namespace, including the debug pod someone `kubectl run`'d last week. Two fixes: give every workload its own SA as a baseline (as this app does), and where pods legitimately share one, close the gap with **workload claims** (§2.7).

In [ ]:
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: allow-checkout, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - from: [{ source: { principals: ["mesh1/ns/petshop/sa/checkout"] } }]
    to:   [{ operation: { ports: ["8080"] } }]
EOF
sleep 10
for d in storefront analytics checkout-blue checkout-green; do
  echo "$d: $(kubectl --context $CTX -n petshop logs deploy/$d --tail=1)"
done
# -> storefront 200, checkout-blue 200, checkout-green 200, analytics still 000


## 2.6 · What else can L4 decide? Namespace, conditions, DENY

Identity is the headline, but ztunnel authorizes on the whole L4 tuple — source **namespace**, source IP, destination **port**, SNI — directly or in a CEL `when` clause, and a `DENY` beats any `ALLOW`. To decide on *where* a caller is, add a caller in a second namespace, `warehouse`, and walk the arc in the Gloo UI Graph: **allow it and watch it appear, then block it and watch it stop.**

In [ ]:
# A second client, in ANOTHER namespace — its identity is spiffe://mesh1/ns/warehouse/sa/warehouse-svc
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: v1
kind: Namespace
metadata:
  name: warehouse
  labels:
    istio.io/dataplane-mode: ambient
---
apiVersion: v1
kind: ServiceAccount
metadata: { name: warehouse-svc, namespace: warehouse }
---
apiVersion: apps/v1
kind: Deployment
metadata: { name: warehouse-svc, namespace: warehouse, labels: { app: warehouse-svc } }
spec:
  replicas: 1
  selector: { matchLabels: { app: warehouse-svc } }
  template:
    metadata: { labels: { app: warehouse-svc } }
    spec:
      serviceAccountName: warehouse-svc
      containers:
        - name: client
          image: curlimages/curl:8.14.1
          command: ["/bin/sh","-c"]
          args:
            - |
              while true; do
                code=$(curl -s -o /dev/null -w '%{http_code}' --max-time 3 http://petstore.petshop:8080/pets || echo 000)
                echo "$(date -u +%H:%M:%S) warehouse-svc -> GET petstore.petshop/pets : $code"
                sleep 2
              done
EOF
kubectl --context $CTX -n warehouse rollout status deploy/warehouse-svc --timeout=90s


In [ ]:
# ONE ALLOW admits callers from TWO namespaces — decided by a CEL `when` on source.namespace
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: l4-allow-petshop-namespace, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - to:   [{ operation: { ports: ["8080"] } }]
    when: [{ key: source.namespace, values: ["petshop", "warehouse"] }]
EOF
sleep 14
echo "petshop:   $(kubectl --context $CTX -n petshop   logs deploy/storefront    --tail=1)"
echo "warehouse: $(kubectl --context $CTX -n warehouse logs deploy/warehouse-svc --tail=1)"
# both 200 — refresh the Graph: warehouse appears, serving petstore


In [ ]:
# Now BLOCK it — same policy, one value removed. warehouse fails closed.
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: l4-allow-petshop-namespace, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - to:   [{ operation: { ports: ["8080"] } }]
    when: [{ key: source.namespace, values: ["petshop"] }]
EOF
sleep 14
echo "warehouse: $(kubectl --context $CTX -n warehouse logs deploy/warehouse-svc --tail=1)"

# the deny as ztunnel logged it — FAIL-CLOSED: allow policies exist, none matched
for i in $(seq 1 12); do
  LINE=$(kubectl --context $CTX -n $ISTIO_NS logs -l app=ztunnel --tail=600 2>/dev/null | grep "policy rejection" | grep warehouse | tail -1)
  [ -n "$LINE" ] && break; sleep 5
done
echo "$LINE" | python3 -c 'import sys,json; l=sys.stdin.read().strip(); d=json.loads(l) if l else {}; print(d.get("src.identity","(no deny line found — the policy may still be converging; re-run this cell)"),"->",d.get("dst.service",""),"\n  error:",d.get("error",""))'


**DENY beats ALLOW.** ztunnel evaluates CUSTOM → DENY → ALLOW, so a `DENY` rule overrides the namespace `ALLOW`. Here `analytics` is in `petshop` (the ALLOW admits it) but a `DENY` on its identity blocks it anyway — a clean carve-out with no change to the broader policy. From the client both denials are the same `000` — the difference is in ztunnel's log: the warehouse deny read `allow policies exist, but none allowed` (fail-closed); this one reads `explicitly denied by: petshop/l4-deny-analytics` — the log **names the policy** that fired.

In [ ]:
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: l4-deny-analytics, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: DENY
  rules:
  - from: [{ source: { principals: ["mesh1/ns/petshop/sa/analytics"] } }]
EOF
sleep 14
for d in storefront analytics; do echo "$d: $(kubectl --context $CTX -n petshop logs deploy/$d --tail=1)"; done

for i in $(seq 1 12); do
  LINE=$(kubectl --context $CTX -n $ISTIO_NS logs -l app=ztunnel --tail=600 2>/dev/null | grep "policy rejection" | grep analytics | tail -1)
  [ -n "$LINE" ] && break; sleep 5
done
echo "$LINE" | python3 -c 'import sys,json; l=sys.stdin.read().strip(); d=json.loads(l) if l else {}; print(d.get("src.identity","(no deny line found — the policy may still be converging; re-run this cell)"),"->",d.get("dst.service",""),"\n  error:",d.get("error",""))'


In [ ]:
# reset the L4-surface policies before closing the gap
kubectl --context $CTX -n petshop delete authorizationpolicy l4-allow-petshop-namespace l4-deny-analytics --ignore-not-found


## 2.7 · Closing the shared-ServiceAccount gap — workload claims

§2.5 hit a wall: two pods on one ServiceAccount are indistinguishable at L4 — one SVID between them. **Workload claims** close it, still at L4, still with no waypoint: with `ENABLE_WORKLOAD_CLAIMS=true`, ztunnel requests a certificate **per pod**, istiod embeds claims in it at issuance, and the `AuthorizationPolicy` matches them with CEL. The mesh is on the 1.30 line, so this is one Helm value on ztunnel — same chart, same version, all the multicluster values kept.

In [ ]:
# ONE new value: ENABLE_WORKLOAD_CLAIMS. Everything else identical to the standup.
helm --kube-context $CTX upgrade -i ztunnel $HREPO/ztunnel -n $ISTIO_NS --version $HVER --wait -f - <<EOF
profile: ambient
hub: $HUB
tag: $TAG
namespace: $ISTIO_NS
istioNamespace: $ISTIO_NS
multiCluster:
  clusterName: mesh1
network: mesh1
platforms:
  peering:
    enabled: true
env:
  LOG_FORMAT: json
  L7_ENABLED: "true"
  SKIP_VALIDATE_TRUST_DOMAIN: "true"
  ENABLE_WORKLOAD_CLAIMS: "true"    # per-POD certs + claim enforcement
EOF
kubectl --context $CTX -n $ISTIO_NS rollout status daemonset/ztunnel --timeout=180s


In [ ]:
# every workload now holds a PER-POD cert (contrast §2.2: one per ServiceAccount)
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=mesh1-worker -o jsonpath='{.items[0].metadata.name}')
$ISTIOCTL --context $CTX ztunnel-config certificates "$ZT.$ISTIO_NS" | grep -E "CERTIFICATE|checkout"


**Annotate the pod — the claim is embedded in its certificate at issuance.** istiod reads `solo.io.security-claims/<key>` off the pod and bakes it into the cert, alongside auto claims for the workload name, namespace and pod. The SPIFFE URI never changes (still `sa/checkout`); the claims ride alongside it.

In [ ]:
# blue is the gold tier, green is silver — one annotation each, pods roll
kubectl --context $CTX -n petshop patch deploy checkout-blue  -p '{"spec":{"template":{"metadata":{"annotations":{"solo.io.security-claims/tier":"gold"}}}}}'
kubectl --context $CTX -n petshop patch deploy checkout-green -p '{"spec":{"template":{"metadata":{"annotations":{"solo.io.security-claims/tier":"silver"}}}}}'
kubectl --context $CTX -n petshop rollout status deploy/checkout-blue deploy/checkout-green --timeout=120s
sleep 5


In [ ]:
# open the certs with openssl and read the claims off the SAN. Each cert gains an
# otherName SAN under Solo's OID (1.3.6.1.4.1.65865.1.1) whose payload is base64url JSON.
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=mesh1-worker -o jsonpath='{.items[0].metadata.name}')
$ISTIOCTL --context $CTX ztunnel-config certificates "$ZT.$ISTIO_NS" -o json > /tmp/zt-certs.json

for pod in checkout-blue checkout-green; do
  jq -r ".[] | select(.identity | contains(\"$pod\")) | .certChain[0].pem" /tmp/zt-certs.json \
    | base64 -d > /tmp/$pod.pem
  echo "── $pod ──────────────────────────────────────────────"
  openssl x509 -in /tmp/$pod.pem -noout -text | grep -A1 "Subject Alternative Name" | tail -1
  openssl x509 -in /tmp/$pod.pem -noout -text | grep -o '65865\.1\.1:.*' | cut -d: -f2 \
    | python3 -c 'import sys,base64,json; p=sys.stdin.read().strip(); print(json.dumps(json.loads(base64.urlsafe_b64decode(p+"="*(-len(p)%4))), indent=2))'
done


Same URI SAN on both (`…/sa/checkout`) — but blue's cert says `tier: gold` and green's says `tier: silver`, signed by istiod. Now authorize on it: swap the SA-wide checkout ALLOW for a claims-scoped one — the same `when` CEL shape as §2.6, over `source.claims` instead of `source.namespace` (the `/` in the annotation key becomes `.` in the claim key). Fail-closed: a workload with no claim never matches the ALLOW.

In [ ]:
kubectl --context $CTX -n petshop delete authorizationpolicy allow-checkout l4-allow-petshop-namespace l4-deny-analytics --ignore-not-found
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata:
  name: allow-gold-checkout
  namespace: petshop
spec:
  selector:
    matchLabels: { app: petstore }
  action: ALLOW
  rules:
    - from:
        - source:
            principals:
              - mesh1/ns/petshop/sa/checkout
      to:
        - operation:
            ports: ["8080"]
      when:
        - key: "source.claims['solo.io.security-claims.tier']"
          values: ["gold"]
EOF
kubectl --context $CTX -n petshop get authorizationpolicy allow-gold-checkout -o jsonpath='{.status.conditions[0].message}'; echo; echo

# fresh per-pod certs + the new policy take ~20-30s to converge
for i in $(seq 1 24); do
  B=$(kubectl --context $CTX -n petshop exec deploy/checkout-blue -- sh -c \
        'curl -s -o /dev/null -w "%{http_code}" --max-time 3 http://petstore:8080/' 2>/dev/null)
  G=$(kubectl --context $CTX -n petshop exec deploy/checkout-green -- sh -c \
        'curl -s -o /dev/null -w "%{http_code}" --max-time 3 http://petstore:8080/' 2>/dev/null)
  echo "  waiting for convergence ($i/24): blue=$B green=$G (want blue=200, green!=200)"
  [ "$B" = "200" ] && [ "$G" != "200" ] && break; sleep 5
done
if [ "$G" = "200" ]; then
  echo "!! green is still allowed — is another ALLOW policy admitting it? kubectl -n petshop get authorizationpolicy"
fi
echo
echo "checkout-blue  (tier gold)   -> $B"
echo "checkout-green (tier silver) -> ${G:-DENIED}   (same sa/checkout, told apart by its cert claim)"


`checkout-blue -> 200`, `checkout-green -> denied` — same ServiceAccount, told apart at L4 by the workload's own certificate. That is the exact thing §2.5 could not do. Note the distinction from what comes next: **this** CEL is over claims in the workload *certificate* at L4; §2.9's CEL is over the user's *request token* at the L7 waypoint.

## 2.8 · Add a waypoint for L7 — and the waypoint is agentgateway

Everything so far was L4 — including the claims. HTTP concerns — methods, paths, JWTs, rate limits — need a **waypoint**, and on Solo Enterprise the waypoint data plane is **agentgateway** (`GatewayClass: enterprise-agentgateway-waypoint`), not Envoy. The division of labour stays clean: ztunnel keeps proving the caller's SPIFFE identity over HBONE at L4, then hands the connection to agentgateway for L7 — and the identity ztunnel proved rides along, available to every policy as `source.identity.*`.

`setup.sh` already installed the agentgateway control plane on both clusters — here we only create the waypoint and enrol the namespace onto it. We reset the L4 policies so the L7 story stands on its own (in production you'd keep both — defence in depth).

In [ ]:
# reset the L4 policies for a clean L7 story (incl. the claims policy)
kubectl --context $CTX -n petshop delete authorizationpolicy allow-storefront allow-checkout allow-gold-checkout --ignore-not-found

# the waypoint: a Gateway of class enterprise-agentgateway-waypoint …
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata:
  name: petshop-waypoint
  namespace: petshop
  labels: { istio.io/waypoint-for: service }
spec:
  gatewayClassName: enterprise-agentgateway-waypoint
  listeners:
  - name: mesh
    port: 15088
    protocol: HTTP
EOF
# … then enrol the whole petshop NAMESPACE onto it (one waypoint for every service)
kubectl --context $CTX label namespace petshop istio.io/use-waypoint=petshop-waypoint --overwrite
sleep 5
kubectl --context $CTX -n petshop rollout status deploy/petshop-waypoint --timeout=150s
# proof it is agentgateway: watch its own request log as the loops flow through
kubectl --context $CTX -n petshop logs deploy/petshop-waypoint --tail=2


## 2.9 · Authorize on the JWT, at L7

Keycloak (installed by `setup.sh`, realm `petshop`, users **alice**/`user` and **bob**/`admin`) issues the tokens. First mint one and read its claims, then two `EnterpriseAgentgatewayPolicy` objects on the waypoint: **`jwtAuthentication` (mode: Strict)** validates every request against Keycloak's JWKS — no token is `401`; **`authorization`** decides with CEL over the validated claims — any valid token may `GET`, `DELETE` additionally requires `realm_access.roles` to contain `admin`.

In [ ]:
# mint + decode a token (run from a petshop pod; it reaches Keycloak cross-namespace)
kubectl --context $CTX -n keycloak rollout status deploy/keycloak --timeout=300s
KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
tok() { kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=$1 -d password=$1 -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p'; }
BOB=$(tok bob)
echo "$BOB" | cut -d. -f2 | { read p; python3 - "$p" <<'PY'
import sys,base64,json
p=sys.argv[1]; p+="="*(-len(p)%4)
d=json.loads(base64.urlsafe_b64decode(p))
print("iss:  ", d["iss"]); print("user: ", d["preferred_username"]); print("roles:", d["realm_access"]["roles"])
PY
}


In [ ]:
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: { name: petshop-jwt, namespace: petshop }
spec:
  targetRefs:
    - { group: gateway.networking.k8s.io, kind: Gateway, name: petshop-waypoint }
  traffic:
    jwtAuthentication:
      mode: Strict
      providers:
        - issuer: http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop
          jwks:
            remote:
              backendRef: { name: keycloak, namespace: keycloak, port: 8080 }
              jwksPath: /realms/petshop/protocol/openid-connect/certs
---
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: { name: petshop-authz, namespace: petshop }
spec:
  targetRefs:
    - { group: gateway.networking.k8s.io, kind: Gateway, name: petshop-waypoint }
  traffic:
    authorization:
      action: Allow
      policy:
        matchExpressions:
          - 'request.method == "GET"'
          - 'request.method == "DELETE" && "admin" in jwt.realm_access.roles'
EOF
sleep 10


In [ ]:
# the matrix: no token, alice (user), bob (admin).
# agentgateway answers a MISSING/invalid token with 401 (authentication),
# and a valid token that fails the CEL with 403 (authorization).
KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
tok() { kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=$1 -d password=$1 -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p'; }
call() { kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c "$1" ; }
ALICE=$(tok alice); BOB=$(tok bob)
echo "no token    GET  /pets     -> $(call "curl -s -o /dev/null -w '%{http_code}' -m5 http://petstore:8080/pets")"
echo "alice       GET  /pets     -> $(call "curl -s -o /dev/null -w '%{http_code}' -m5 -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets")"
echo "alice(user) DELETE /pets/1 -> $(call "curl -s -o /dev/null -w '%{http_code}' -m5 -X DELETE -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets/1")"
echo "bob(admin)  DELETE /pets/1 -> $(call "curl -s -o /dev/null -w '%{http_code}' -m5 -X DELETE -H 'Authorization: Bearer $BOB' http://petstore:8080/pets/1")"


**Look at the Graph now — the denials are RED.** An L7 denial is a real HTTP response, so the Graph can finally draw a block: red edges from every (token-less) client loop into `petshop-waypoint`. Contrast the L4 sections, where a denial is a reset connection and a block only ever showed as an edge going quiet. L4 deny = silence; L7 deny = red edge with a status code.

## 2.10 · Route at the waypoint — canary and header shift

The waypoint is not just a policy point — agentgateway is a standard **Gateway API** data plane, so the same proxy that checks the JWT also routes. Attach an `HTTPRoute` whose `parentRef` is the petstore **Service** (the GAMMA pattern): callers keep calling `petstore:8080`, and the waypoint enforces a **90/10 canary** to `petstore-v2` plus a header shift (`x-beta: true` goes straight to v2). The JWT policy still gates **every** request, whichever version it lands on — policy and routing compose.

In [ ]:
# petstore-v2: the same app (it reports served_by), the SAME ServiceAccount
# (routing does not change identity), its own Service.
kubectl --context $CTX apply -f yaml/50-l7/30-petstore-v2.yaml
kubectl --context $CTX -n petshop rollout status deploy/petstore-v2 --timeout=120s

kubectl --context $CTX apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata:
  name: petstore-split
  namespace: petshop
spec:
  parentRefs:
    - group: ""
      kind: Service
      name: petstore
      port: 8080
  rules:
    - matches:
        - headers:
            - name: x-beta
              value: "true"
      backendRefs:
        - name: petstore-v2
          port: 8080
    - backendRefs:
        - name: petstore
          port: 8080
          weight: 90
        - name: petstore-v2
          port: 8080
          weight: 10
EOF
kubectl --context $CTX -n petshop get httproute petstore-split \
  -o jsonpath='{.status.parents[0].conditions[?(@.type=="Accepted")].status}{" — accepted by Service/"}{.status.parents[0].parentRef.name}'; echo
sleep 10


In [ ]:
# the proof: distribution, header shift, and the JWT policy still in force
KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
ALICE=$(kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=alice -d password=alice -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p')

echo "— 20 GETs with alice's token: the canary split —"
kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "for i in \$(seq 1 20); do curl -s -m5 -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets; echo; done" \
  | grep -o '"served_by": "[^"]*"' | sed 's/.*: "//; s/-[a-z0-9]*-[a-z0-9]*"$//' | sort | uniq -c

echo "— 5 GETs with x-beta: true -> always v2 —"
kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "for i in \$(seq 1 5); do curl -s -m5 -H 'Authorization: Bearer $ALICE' -H 'x-beta: true' http://petstore:8080/pets; echo; done" \
  | grep -o '"served_by": "[^"]*"' | sed 's/.*: "//; s/-[a-z0-9]*-[a-z0-9]*"$//' | sort | uniq -c

echo "— no token, even on the beta route —"
kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -o /dev/null -w 'no token -> %{http_code}\n' -m5 -H 'x-beta: true' http://petstore:8080/pets"


## 2.11 · Rate limit — by workload identity

The user's token says who the **user** is; the certificate says who the **workload** is. Here both meet: a rate limit whose CEL condition keys on `source.identity.serviceAccount` — the SPIFFE identity ztunnel proved at L4 — enforced locally in the waypoint. Only `storefront` draws from this bucket; `checkout` presenting the **same user JWT** is untouched.

In [ ]:
# 5 requests/minute for the storefront IDENTITY, everyone else unlimited
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: { name: petshop-ratelimit-storefront, namespace: petshop }
spec:
  targetRefs:
    - { group: gateway.networking.k8s.io, kind: Gateway, name: petshop-waypoint }
  traffic:
    rateLimit:
      conditional:
        - condition: 'source.identity.serviceAccount == "storefront"'
          policy:
            local:
              - requests: 5
                unit: Minutes
EOF
sleep 8

KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
ALICE=$(kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=alice -d password=alice -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p')

echo "— storefront, 8 rapid GETs (alice's token) —"
kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "for i in \$(seq 1 8); do curl -s -o /dev/null -w '%{http_code} ' -m5 -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets; done"; echo

echo "— checkout-blue, 8 rapid GETs (the SAME token) —"
kubectl --context $CTX -n petshop exec deploy/checkout-blue -- sh -c \
  "for i in \$(seq 1 8); do curl -s -o /dev/null -w '%{http_code} ' -m5 -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets; done"; echo


`storefront: 200 ×5 then 429 429 429` — `checkout-blue: 200 ×8`, on the **same token**. The rate budget followed the workload's certificate, not the user's JWT. That is the identity thesis of this whole part, applied to L7 traffic management.

## 2.R · Reset Part 2

Removes the petshop policies, waypoint, routes and v2, reverts ztunnel to claims-off (so §2.2's four-certs moment works on a re-run), and drops the claim annotations. The petshop itself stays deployed.

In [ ]:
kubectl --context $CTX -n petshop delete enterpriseagentgatewaypolicy petshop-jwt petshop-authz petshop-ratelimit-storefront --ignore-not-found
kubectl --context $CTX -n petshop delete httproute petstore-split --ignore-not-found
kubectl --context $CTX -n petshop delete deploy petstore-v2 --ignore-not-found; kubectl --context $CTX -n petshop delete svc petstore-v2 --ignore-not-found
kubectl --context $CTX label namespace petshop istio.io/use-waypoint- 2>/dev/null || true
kubectl --context $CTX -n petshop delete gateway petshop-waypoint --ignore-not-found
kubectl --context $CTX -n petshop delete authorizationpolicy allow-storefront allow-checkout allow-gold-checkout l4-allow-petshop-namespace l4-deny-analytics --ignore-not-found
kubectl --context $CTX delete ns warehouse --ignore-not-found
kubectl --context $CTX -n petshop patch deploy checkout-blue  --type=json -p='[{"op":"remove","path":"/spec/template/metadata/annotations/solo.io.security-claims~1tier"}]' 2>/dev/null || true
kubectl --context $CTX -n petshop patch deploy checkout-green --type=json -p='[{"op":"remove","path":"/spec/template/metadata/annotations/solo.io.security-claims~1tier"}]' 2>/dev/null || true

# ztunnel back to claims-off (same values as the standup)
helm --kube-context $CTX upgrade -i ztunnel $HREPO/ztunnel -n $ISTIO_NS --version $HVER --wait -f - <<EOF
profile: ambient
hub: $HUB
tag: $TAG
namespace: $ISTIO_NS
istioNamespace: $ISTIO_NS
multiCluster:
  clusterName: mesh1
network: mesh1
platforms:
  peering:
    enabled: true
env:
  LOG_FORMAT: json
  L7_ENABLED: "true"
  SKIP_VALIDATE_TRUST_DOMAIN: "true"
EOF
kubectl --context $CTX -n $ISTIO_NS rollout status daemonset/ztunnel --timeout=180s
echo "Part 2 reset."
